# Generate a forecast with AIFS Single v2

**Author**: ECMWF AIFS team

*This notebook was last tested and operational on 19/06/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was adapted for the DestinE [2026 Machine Learning for Earth System Modelling Course](https://learning.ecmwf.int/course/view.php?id=99) from the published [AIFS v2 page on HuggingFace](https://huggingface.co/ecmwf/aifs-single-2.0/blob/main/run_AIFS_v2.0.ipynb).

It demonstrates how to generate a global weather forecast with AIFS Single v2 using ECMWF [open data](https://www.ecmwf.int/en/forecasts/datasets/open-data), [anemoi-inference](https://anemoi-inference.readthedocs.io/en/latest/), and a HuggingFace model checkpoint. The `aifs_single_v2.0.ckpt` checkpoint contains model weights only and cannot be used for fine-tuning.
<!-- ::: -->

<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
You may try to run/access this notebook on the free online platforms linked below. Please note they are not officially supported by or linked with ECMWF.

**Important note:**
This notebook requires specific GPU access and several GB of writeable disk space. It has been tested on Colab with **L4 and A100** GPUs. These do not come with Google's free plan, and such GPUs are also not available for free on Binder. Therefore, if you wish to run this notebook yourself, the best thing to do is find your own suitable GPU-based system, and set it up and run it there. If you have access to (for example) GPUs on ECMWF's ATOS HPCF, you can run this notebook through ECMWF's [JupyterHub](https://jupyterhub.ecmwf.int/).  Depending on your system, small additional adjustments may be needed to run the notebook, and we cannot help you with this.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/m4/run_AIFS_v2.0.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m4/run_AIFS_v2.0.ipynb)
[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/2026-ml-esm-training/main?binder_path=m4&labpath=m4/run_AIFS_v2.0.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m4/run_AIFS_v2.0.ipynb)
<!-- 
::: -->


# Introduction

In this session, we will:

* Load meteorological open-source data from ECMWF at different heights
* Define initial conditions for the model
* Generate a real-world AI weather forecast using AIFS from ECMWF
* Inspect and play with the forecast (optional)

## Prepare your environment

This notebook requires the following packages:
- Python (version 3.11 or 3.12)
- numpy
- matplotlib
- cartopy
- anemoi-inference[huggingface] (version=0.8.3)
- anemoi.graphs (version=0.6.4)
- torch (version=2.7.0)
- torch-geometric (version=2.6.1)
- anemoi-models (version=0.9.3)
- anemoi-utils (version=0.4.33)
- anemoi-datasets (version=0.5.26)
- earthkit-regrid (version=0.5.1)
- ecmwf-opendata (version=0.3.29)
- earthkit-data (version <1.0.0)

This notebook also requires the following:
- Ampere GPUs or newer (this notebook has been tested in Colab using the **L4** and **A100** runtimes, and on ECMWF's ATOS HPCF **AC cluster**)
- Several GB of writable disk space

Known limitations:
- Even with compatible GPUs, this notebook may not work out-of-the-box on all systems. A compataible CUDA/PyTorch/FlashAttention environment is required. See https://github.com/Dao-AILab/flash-attention?tab=readme-ov-file#installation-and-features for more details.

# 1. Install dependencies

Run the lines below to install the required packages.

### To run this notebook in Colab (L4 or A100 runtimes)

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/m4/requirements.txt

# Install flash-attn from a pre-built wheel (no compilation required).
# Pre-built wheels are available for Python 3.11 and 3.12 with CUDA 12 and torch 2.7 on Linux x86_64.
# For other configurations we try to build from source, which requires the CUDA toolkit (nvcc).
import sys, subprocess

_base = "https://github.com/cathalobrien/get-flash-attn/releases/download/v0.1-alpha"
_wheels = {
    (3, 11): f"{_base}/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiFALSE-cp311-cp311-linux_x86_64.whl",
    (3, 12): f"{_base}/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiFALSE-cp312-cp312-linux_x86_64.whl",
}
_ver = sys.version_info[:2]
_wheel = _wheels.get(_ver)
if _wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    f"flash-attn @ {_wheel}"], check=True)
else:
    print(f"No pre-built wheel for Python {_ver[0]}.{_ver[1]} — building from source (requires nvcc)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn==2.7.4.post1",
                    "--no-build-isolation"], check=True)

### To run this notebook through JupyterHub on ECMWF's ATOS HPCF
Uncomment the lines below to install the requirements in your active environment.

In [ ]:
# %pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/m4/requirements.txt
# %pip install -q --force-reinstall --no-deps "flash-attn @ https://github.com/cathalobrien/get-flash-attn/releases/download/v0.1-alpha/flash_attn-2.8.3+cu12torch2.7cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

# 2. Check runtime environment

The code below checks whether a CUDA gpu is available which is required to run the AIFS model later on. In case CUDA available is **False** then later on in the notebook you might run into processing problems

In [ ]:
import platform
import torch

print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. This notebook requires a CUDA-capable GPU.\n"
        "  • On Colab: Runtime → Change runtime type → select L4 or A100 GPU.\n"
        "  • On ATOS HPCF: ensure you have requested GPU resources (AC cluster)."
    )

# See https://github.com/huggingface/transformers/issues/28188
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)

print("GPU:", gpu_name)
print(f"Compute capability: {major}.{minor}")

if major < 8:
    raise RuntimeError(
        f"FlashAttention requires Ampere (compute capability >= 8.0) or newer.\n"
        f"Your GPU ({gpu_name}) has compute capability {major}.{minor}.\n"
        "  • On Colab: switch to an L4 or A100 runtime."
    )

In [ ]:
try:
    import flash_attn
    print("FlashAttention:", flash_attn.__version__)
except ImportError:
    raise RuntimeError(
        "flash-attn is not installed. Re-run the install cell (Section 1) and restart the kernel.\n"
        "Ensure you are on a compatible GPU node before installing."
    )

# 3. Import packages

In [ ]:
import datetime
from collections import defaultdict

import numpy as np

import earthkit.data as ekd
import earthkit.regrid as ekr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from ecmwf.opendata import Client as OpendataClient

Store downloaded ECMWF data in local cache:


In [ ]:
ekd.config.set({'cache-policy': 'user'})

# 4. Prepare retrieval of initial conditions

Initial conditions are the initial state used by the model.

### List parameters to retrieve from ECMWF open data

Below are the variables and levels listed which data is used. These variables are from the Surface, Soils, and Oceans and are used to make a weather forecast

In [ ]:
PARAM_SFC = ["10u", "10v", "2d", "2t", "msl", "skt", "sp", "tcw", "lsm", "z", "slor", "sdor", "sd"]
PARAM_SOIL =["vsw","sot"]
PARAM_WAVE =["wmb", "h1012", "h1214", "h1417", "h1721", "h2125", "h2530", "mwd", "cdww", "mwp", "swh"]
PARAM_PL  = ["gh", "t", "u", "v", "q"]
LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50, 10]
SOIL_LEVELS = [1,2]

### Choose open data source and set the latest available date

In [ ]:
SOURCE = "ecmwf" # Other options are: "azure", "aws", or "google"

DATE = OpendataClient(SOURCE).latest()
print("Initial date is", DATE)

### Create function to download and interpolate data from the ECMWF Open Data API

In [ ]:
def get_open_data(param, levelist=[], **kwargs):
    fields = defaultdict(list)
    # Get the data for the current date and the previous date as the model is initialised with t-12h data
    for date in [DATE - datetime.timedelta(hours=12), DATE]:
        data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist, source = SOURCE, **kwargs)
        
        for f in data: # type: ignore
            # Open data is between -180 and 180, we need to shift it to 0-360
            assert f.to_numpy().shape == (721,1440)
            values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
            # Interpolate the data to from 0.25 to N320
            values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            # Add the values to the list
            name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levelist else f.metadata("param")
            fields[name].append(values)

    # Create a single matrix for each parameter
    for param, values in fields.items():
        fields[param] = np.stack(values)

    return fields

### Store model input fields

In [ ]:
fields = {}

# 5. Download initial conditions from ECMWF open data

Downloads can take several minutes and may be interrupted during peak times.

### Download surface fields

In [ ]:
fields.update(get_open_data(param=PARAM_SFC, levtype="sfc"))
assert all(p in fields for p in PARAM_SFC), "Missing parameters: %s" % (set(PARAM_SFC) - set(fields.keys()))

### Download wave fields

In [ ]:
fields.update(get_open_data(param=PARAM_WAVE, stream="wave"))
assert all(p in fields for p in PARAM_WAVE), "Missing parameters: %s" % (set(PARAM_WAVE) - set(fields.keys()))

### Download soil fields

In [ ]:
soil=get_open_data(param=PARAM_SOIL,levelist=SOIL_LEVELS)

soil_names = [f"{p}_{lev}" for p in PARAM_SOIL for lev in SOIL_LEVELS]
assert all(p in soil for p in soil_names), "Missing parameters: %s" % (set(soil_names) - set(soil.keys()))

### Download pressure level fields

In [ ]:
fields.update(get_open_data(param=PARAM_PL, levelist=LEVELS))

PRESSURE_NAMES = [f"{p}_{lev}" for p in PARAM_PL for lev in LEVELS]
assert all(p in fields for p in PRESSURE_NAMES), "Missing parameters: %s" % (set(PRESSURE_NAMES) - set(fields.keys()))

## Placing the AIFS in context of earlier notebooks

Before passing this data to the model, let us quickly re-establish what AIFS is and what it expects as input. AIFS Single v2 brings together the two ideas you met in the module 2 notebooks:

- Like the Graph Neural Network notebook, it represents the atmosphere as values on a reduced Gaussian grid (here N320, ≈ 0.25°, ~542,080 points) and uses a graph encoder and decoder to move between this grid and an internal latent representation.
- Like the Transformer notebook, the latent state is then processed into another one by a transformer ("the processor"), whose attention lets distant parts of the globe influence one another in the forecast.

Also as in the Transformer notebook, AIFS forecasts *one step at a time*: it predicts the state 6 hours ahead, then feeds that prediction back in to produce the next 6-hour step. A 12-hour forecast is therefore two model steps. This is called an autoregressive forecast.

## Input format to the AIFS

### Input from two time steps
Look at the shape of any field below. It is `(2, 542080)`, not `(542080,)`. The model is given the atmosphere at **two times**: the analysis time and an earlier time (see the `get_open_data` function which time lag is used). This lets it infer *tendencies* — not just *where* a weather system is, but how fast and in which direction it is moving.

### Variables and levels
AIFS is driven by a fixed set of variables we just collected in `fields`: surface fields (e.g. `2t`, `msl`, `10u`/`10v`), soil fields, ocean wave fields, and upper-air fields on pressure levels. The upper-air fields use the naming convention `{variable}_{level}` — e.g. `t_500` is temperature at 500 hPa. The 14 levels in `LEVELS` run from 1000 hPa near the surface up to 10 hPa in the stratosphere, giving the model a full 3D picture of the atmosphere.

## Task: look inside the model's input

The upper-air fields span 14 pressure levels. Plot the global-mean vertical profile of temperature (`t_*`) and specific humidity (`q_*`) against pressure (`LEVELS`) for the analysis time (i.e. the second of the two time steps). What vertical structure do you see, and does it match your intuition about the atmosphere?

In [ ]:
import matplotlib.pyplot as plt

# Global-mean vertical profiles at the analysis time (time index 1 = t=0).
#    Each t_{lev} / q_{lev} array has shape (2, n_gridpoints); we average over the grid.
t_profile =    # K -> °C
q_profile =    # kg/kg -> g/kg

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

ax1.plot(t_profile, LEVELS, "o-")
ax1.set_xlabel("Temperature (°C)")
ax1.set_ylabel("Pressure (hPa)")
ax1.set_title("Global-mean temperature")
ax1.grid(True, alpha=0.3)

ax2.plot(q_profile, LEVELS, "o-", color="C1")
ax2.set_xlabel("Specific humidity (g/kg)")
ax2.set_title("Global-mean specific humidity")
ax2.grid(True, alpha=0.3)

ax1.set_yscale("log")
ax1.invert_yaxis()   # surface (1000 hPa) at the bottom, top of atmosphere up high
plt.tight_layout()
plt.show()

# 6. Apply data transformations

AIFS Single v2 expects the data to match the data it was trained on.

Transform the mean wave direction into sine and cosine components:

In [ ]:
mwd = fields.pop("mwd")
mwd_rad = np.deg2rad(mwd)

fields["cos_mwd"] = np.cos(mwd_rad)
fields["sin_mwd"] = np.sin(mwd_rad)

Rename soil fields:

In [ ]:
mapping = {'sot_1': 'stl1', 'sot_2': 'stl2',
           'vsw_1': 'swvl1','vsw_2': 'swvl2'}
for k,v in soil.items():
    fields[mapping[k]]=v

In [ ]:
print(fields['u_10'])
print(fields['10u'])

Some specific humidity fields are not used for the AIFS and need to be removed

In [ ]:
fields.pop("q_10", None)  # Remove the 10hPa level for specific humidity, as it is not used in the model
fields.pop("q_50", None);  # Remove the 50hPa level for specific humidity, as it is not used as a prognostic in the model

Apply land-sea mask:

In [ ]:
#mask for values where lsm=0
mask = fields["lsm"][0].flatten() == 0
fields["sd"][:, mask] = np.nan
fields["swvl1"][:, mask] = np.nan
fields["swvl2"][:,mask] = np.nan

Convert geopotential height into geopotential:

In [ ]:
# Transform GH to Z
for level in LEVELS:
    gh = fields.pop(f"gh_{level}")
    fields[f"z_{level}"] = gh * 9.80665

# 7. Create initial forecast state

In [ ]:
input_state = dict(date=DATE, fields=fields)

# 8. Load AIFS Single v2 model

### Download the model checkpoint from Hugging Face

In [ ]:
checkpoint = {"huggingface":"ecmwf/aifs-single-2.0"}

To reduce the memory usage of the model one can set certain environment variables, like the number of chunks of the model's mapper.
Please refer to:
- https://anemoi.readthedocs.io/projects/models/en/latest/modules/layers.html#anemoi-inference-num-chunks
- https://pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf

for more information. To do so, use the code below:
```
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True' 
os.environ['ANEMOI_INFERENCE_NUM_CHUNKS']='16'
```

### Create a runner

In [ ]:
runner = SimpleRunner(checkpoint, device='cuda')

**Note - changing the device from GPU to CPU**

- Running the transformer model used on the CPU is tricky, it depends on the FlashAttention library which only supports Nvidia and AMD GPUs, and is optimised for performance and memory usage
- In newer versions of anemoi-models, v0.4.2 and above, there is an option to switch off flash attention and uses Pytorchs Scaled Dot Product Attention (SDPA). The code snippet below shows how to overwrite a model from a checkpoint to use SDPA. Unfortunately it's not optimised for memory usage in the same way, leading to much greater memory usage. Please refer to https://github.com/ecmwf/anemoi-inference/issues/119 for more details 

# 9. Run the model to generate a forecast

The example below is a 12-hour forecast for demonstration purposes. Change the LEAD_TIME to modify the forecast length.

In [ ]:
print(input_state.keys())
print(fields.keys())

In [ ]:
LEAD_TIME = 12
states = []
for state in runner.run(input_state=input_state, lead_time=LEAD_TIME):
    states.append(state)
    print_state(state)

### Note
Users should not expect this notebook to reproduce ECMWF operational AIFS forecasts exactly. This is due to two factors:

#### 1. GPU non-determinism
GPU operations are not always bitwise deterministic, meaning repeated runs can produce slightly different numerical results.

To enforce determinism at GPU level, the following settings can be configured:

```
#First, in a terminal
export CUBLAS_WORKSPACE_CONFIG=:4096:8

#And then before running inference:
import torch
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

```
Please note that using the above approach to enable deterministic behaviour will **significantly** increase runtime. 

#### 2. Differences in input data reprojection 
The initial conditions used in this notebook are downloaded from ECMWF open data. The data is provided on a 0.25° latitude/longitude grid and reprojected to the N320 grid for use by the AIFS.

In ECMWF's operational forecasting system, the initial conditions come directly from operational IFS analyses on the native o1280 grid and are reprojected directly to N320.

These differing reprojection pathways can introduce small differences in the model input fields, which may lead to minor differences in the resulting forecasts.

# 10. Optional: Inspect the forecast

#### Plot a field (e.g. 100u)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as tri

In [ ]:
def fix(lons):
    # Shift longitudes from 0–360 to −180–180
    return np.where(lons > 180, lons - 360, lons)

latitudes  = states[-1]["latitudes"]
longitudes = states[-1]["longitudes"]
values     = states[-1]["fields"]["100u"]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=":")

triangulation = tri.Triangulation(fix(longitudes), latitudes)

contour = ax.tricontourf(triangulation, values, levels=20, transform=ccrs.PlateCarree(), cmap="RdBu")
cbar = fig.colorbar(contour, ax=ax, orientation="vertical", shrink=0.7, label="100 m zonal wind (m/s)")

plt.title(f"100 m zonal wind (100u) at {states[-1]['date']}")
plt.show()

## Task: Make a forecast for your own region

The map above is global. Now let's zoom in on a region you care about and build a small weather chart from the forecast.

1. Define a latitude/longitude bounding box for your region (the example below uses Europe).
2. Pick a set of variables to show together. The example shades 2 m temperature (`2t`), with mean-sea-level pressure (`msl`) drawn as isobars.

**Bonus:** increase `LEAD_TIME` in Section 9 (e.g. to 72 for a 3-day forecast), re-run the model, and watch how the systems over your region evolve.

In [ ]:
# Define your region (example: Europe)
LAT_MIN, LAT_MAX = 35, 72
LON_MIN, LON_MAX = -15, 30

final = states[-1]
lats = final["latitudes"]
lons = fix(final["longitudes"])

# Select the grid points that fall inside the box
sel = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
region = tri.Triangulation(lons[sel], lats[sel])

t2m  = final["fields"]["2t"][sel] - 273.15   # K -> °C
mslp = final["fields"]["msl"][sel] / 100.0   # Pa -> hPa

fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
ax.coastlines(resolution="50m")
ax.add_feature(cfeature.BORDERS, linestyle=":")

# 2 m temperature, shaded
cf = ax.tricontourf(region, t2m, levels=20, cmap="RdBu_r", transform=ccrs.PlateCarree())
fig.colorbar(cf, ax=ax, shrink=0.8, label="2 m temperature (°C)")

# Mean-sea-level pressure, drawn as isobars
cs = ax.tricontour(region, mslp, levels=12, colors="k", linewidths=0.7,
                   transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=8, fmt="%d")

ax.set_title(f"AIFS forecast valid {final['date']}")
plt.show()